In [1]:
# ---------------------------------------------------------
# Cell 1: Import Libraries
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import ipywidgets as widgets
from IPython.display import display, clear_output
import urllib.request
import zipfile
import os
import warnings
warnings.filterwarnings('ignore') # Keeps our presentation clean from red warning text

print("Step 1 Complete: All libraries imported successfully!")

Step 1 Complete: All libraries imported successfully!


In [4]:
# ---------------------------------------------------------
# Cell 2: Download and Load the MovieLens Dataset
# ---------------------------------------------------------
def fetch_movielens_data():
    """Downloads and extracts the dataset if it doesn't already exist."""
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    zip_path = "ml-100k.zip"
    
    if not os.path.exists("ml-100k"):
        print("Downloading MovieLens 100k dataset (this only happens once)...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
        print("Download complete.")
    else:
        print("Dataset found locally.")

fetch_movielens_data()

# Read the data files into pandas DataFrames
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'], encoding='latin-1')
users = pd.read_csv('ml-100k/u.user', sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code'], encoding='latin-1')
movie_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 
              'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=movie_cols, encoding='latin-1')

# Combine them into one master dataset
data = pd.merge(pd.merge(ratings, users, on='user_id'), movies, on='movie_id')

print(f"Step 2 Complete: Loaded {len(data)} real-world movie ratings!")

Download complete.
Step 2 Complete: Loaded 100000 real-world movie ratings!


In [6]:
# ---------------------------------------------------------
# Cell 3: Model Training and Preprocessing
# ---------------------------------------------------------
print("Preparing data and training the model...")

# 1. Feature Engineering: Convert text to numbers
data['gender_num'] = data['gender'].map({'M': 1, 'F': 0})

# Calculate averages to give the model context about users and movies
user_avg = data.groupby('user_id')['rating'].mean().reset_index().rename(columns={'rating': 'user_avg_rating'})
movie_avg = data.groupby('movie_id')['rating'].mean().reset_index().rename(columns={'rating': 'movie_avg_rating'})

data = pd.merge(data, user_avg, on='user_id')
data = pd.merge(data, movie_avg, on='movie_id')

# 2. Define the exact features we want the model to learn from
features = ['user_id', 'movie_id', 'age', 'gender_num', 'user_avg_rating', 'movie_avg_rating', 
            'Action', 'Comedy', 'Drama', 'Sci-Fi', 'Romance']

X = data[features]
y = data['rating']

# 3. Train the model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

# 4. Prepare a "Movie Catalog" DataFrame for our live UI to use later
# This contains just the unique movies and their specific features
movie_catalog = data[['movie_id', 'title', 'movie_avg_rating', 'Action', 'Comedy', 'Drama', 'Sci-Fi', 'Romance']].drop_duplicates()

print(f"Step 3 Complete! Model trained successfully.")
print(f"Error margin (RMSE): {rmse:.2f} stars.")

Preparing data and training the model...
Step 3 Complete! Model trained successfully.
Error margin (RMSE): 0.92 stars.
